# Шаг 2. Генерация (RuBQ 2.0, проверка предпосылки)


In [ ]:
SMOKE = True   # True: 5 вопросов, файл generations_smoke.jsonl; False: все 300

In [ ]:
!pip install -q -U bitsandbytes
!git clone -q --depth 1 https://github.com/vladislavneon/RuBQ.git /kaggle/working/RuBQ || true

In [ ]:
import json, random, os, time, re
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers, bitsandbytes
print(torch.__version__, transformers.__version__, bitsandbytes.__version__, torch.cuda.get_device_name(0))

SEED = 42
N = 300
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUT = "/kaggle/working/" + ("generations_smoke.jsonl" if SMOKE else "generations.jsonl")

data = json.load(open("/kaggle/working/RuBQ/RuBQ_2.0/RuBQ_2.0_test.json"))
answerable = sorted([q for q in data if q["answers"]], key=lambda q: q["uid"])
assert len(answerable) == 1920, len(answerable)
sample = random.Random(SEED).sample(answerable, N)

assert [q["uid"] for q in sample[:10]] == [6432, 308, 66, 6689, 734, 662, 604, 382, 6678, 283]
if SMOKE:
    sample = sample[:5]
print(len(sample), "вопросов →", OUT)

In [ ]:
PROMPT_VERSION = "v1"
SYSTEM = ("Ты отвечаешь на вопросы викторины. Ответь кратко: только сущность, число или дата. "
          "Без пояснений, без полного предложения.")
FEWSHOT = [
    ("Какой город является столицей Японии?", "Токио"),
    ("Сколько планет в Солнечной системе?", "8"),
    ("Кто написал роман «Преступление и наказание»?", "Фёдор Достоевский"),
    ("Когда был основан Санкт-Петербург?", "1703"),
]
def build_messages(question):
    msgs = [{"role": "system", "content": SYSTEM}]
    for q, a in FEWSHOT:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": question})
    return msgs

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})
model.eval()

GREEDY = dict(max_new_tokens=16, do_sample=False, num_beams=1, repetition_penalty=1.0,
              temperature=None, top_p=None, top_k=None)
SAMPLE = dict(max_new_tokens=16, do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
              repetition_penalty=1.0, num_return_sequences=10)
_eos = model.generation_config.eos_token_id
EOS_IDS = torch.tensor(_eos if isinstance(_eos, list) else [_eos], device=model.device)
print(model.generation_config, "EOS:", EOS_IDS.tolist())

In [ ]:
def run_one(q):
    prompt = tok.apply_chat_template(build_messages(q["question_text"]), tokenize=False,
                                     add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    L = enc["input_ids"].shape[1]
    with torch.no_grad():
        g = model.generate(**enc, **GREEDY, output_scores=True, return_dict_in_generate=True,
                           pad_token_id=tok.eos_token_id)
        gen_ids = g.sequences[0, L:]
        # логвероятности токенов жадного ответа
        lp = model.compute_transition_scores(g.sequences, g.scores, normalize_logits=True)[0]
        keep = ~torch.isin(gen_ids, EOS_IDS)
        greedy_logprobs = lp[keep].float().cpu().tolist()
        greedy_ntok = int(keep.sum())
        seed = SEED * 100000 + q["uid"]
        torch.manual_seed(seed)
        s = model.generate(**enc, **SAMPLE, pad_token_id=tok.eos_token_id)
    samples = [tok.decode(x[L:], skip_special_tokens=True).strip() for x in s]
    samples_ntok = [int((~torch.isin(x[L:], EOS_IDS)).sum()) for x in s]
    return {
        "uid": q["uid"], "question": q["question_text"],
        "gold_labels": [a.get("label") for a in q["answers"]],
        "prompt": prompt, "prompt_version": PROMPT_VERSION, "model": MODEL_ID, "quant": "bnb-nf4-4bit",
        "greedy": tok.decode(gen_ids, skip_special_tokens=True).strip(),
        "greedy_ntok": greedy_ntok, "greedy_logprobs": greedy_logprobs,
        "samples": samples, "samples_ntok": samples_ntok,
        "gen_greedy": {k: v for k, v in GREEDY.items() if v is not None},
        "gen_sample": SAMPLE, "seed": seed,
    }

done = set()
if os.path.exists(OUT):
    done = {json.loads(l)["uid"] for l in open(OUT)}
todo = [q for q in sample if q["uid"] not in done]
print("готово", len(done), "осталось", len(todo))
t0 = time.time()
with open(OUT, "a") as f:
    for i, q in enumerate(todo):
        f.write(json.dumps(run_one(q), ensure_ascii=False) + "\n"); f.flush()
        if i % 20 == 0:
            print(i, f"{time.time()-t0:.0f}s", q["question_text"][:60])
print("всего сек:", round(time.time() - t0))

In [ ]:
rows = [json.loads(l) for l in open(OUT)]
for r in rows[:5]:
    print(r["uid"], "|", r["question"], "| эталон:", r["gold_labels"])
    print("   greedy:", repr(r["greedy"]), f"({r['greedy_ntok']} tok)")
    print("   samples:", r["samples"])
all_ntok = [r["greedy_ntok"] for r in rows] + [n for r in rows for n in r["samples_ntok"]]
texts = [r["greedy"] for r in rows] + [s for r in rows for s in r["samples"]]
print(f"\nстрок: {len(rows)}")
print(f"уперлись в 16 токенов: {sum(n >= 16 for n in all_ntok) / len(all_ntok):.1%} "
      f"(жадные: {sum(r['greedy_ntok'] >= 16 for r in rows) / len(rows):.1%})")
print(f"без кириллицы и цифр: {sum(not re.search(r'[А-Яа-яЁё0-9]', t) for t in texts) / len(texts):.1%}")
print(f"больше 5 слов: {sum(len(t.split()) > 5 for t in texts) / len(texts):.1%}")